In [1]:
############################################################
# Headers

import os
import re
import configparser
from   glob import glob

from scipy.optimize    import curve_fit, fsolve, root
from scipy.interpolate import interp1d, interp2d, interpn

from   pylab      import *
import matplotlib
from   matplotlib.colors import Normalize, LogNorm

############################################################
# Constants

h      = 6.62607e-27;   # CGS Planctk constant
kb     = 1.38065e-16;   # CGS Boltzmann constant
eV     = 1.60218e-12;   # CGS eV
c      = 2.99792458e10; # CGS speed of light
q_e    = 4.80321e-10;   # CGS electron charge
me     = 9.1094e-28;    # CGS electron mass;
mp     = 1.67262e-24;   # CGS proton mass;
NA     = 6.022141e23    # Avogadro constant
ma     = 1 / NA;        # Atomic mass unit
AU     = 1.49598e13     # Astronomical Unit in cm
yr     = 3.1536e7       # 1 yr in sec
G      = 6.6742831e-8   # CGS graviational constant
sig_sb = 5.6704e-5      # Stefan-Boltzmann
yr     = 365. * 86400.; # Year in secons

modot  = 1.9891e33;     # Solar mass
rodot  = 6.96e10;       # Solar radius
lodot  = 3.828e33;      # Solar luminosity
mearth = 5.9742e27      # Earth mass
rearth = 6.37814e8;     # Earth Radius

############################################################

In [2]:
############################################################
# Basic chemistry functions;
##############################
elem_data = {  };
with open( 'elem_table.txt' ) as f:
    for line in f.readlines(   ) :
        l = line.strip(  ).split(  );
        elem_data[ l[ 1 ] ] \
            = ( int( l[ 0 ] ), int  ( l[ 2 ] )  ,
                int( l[ 3 ] ), float( l[ 4 ] ) );
#

def species_to_elem( s ):
    if s == 'e-' or 'Gr' in s:
        return {  };
    #
    pe   = re.compile( r'[A-Z][a-z]*(?=\d*)' );
    pd   = re.compile( r'^\d+' );
    res  = { 'charge' : 0 };
    if '+' in s:
        np = s.split( '+' )[ 1 ];
        res[ 'charge' ] += \
            1 if len( np ) == 0 else int( np );
    if '-' in s:
        nn = s.split( '-' )[ 1 ];
        res[ 'charge' ] -= \
            1 if len( nn ) == 0 else int( nn );
    #
    s = s.replace( '+', '' );
    s = s.replace( '-', '' );
    while s != '':
        e = pe.search( s );
        e = e.group(  );
        s = s.replace( e, '', 1 );
        d = pd.search( s );
        if d != None:
            d = d.group(  );
            s = s.replace( d, '', 1 );
        else:
            d = '1';
        #
        if not e in res:
            res[ e ]  = int( d );
        else:
            res[ e ] += int( d );
        #
    #
    return res;
#
############################################################

In [5]:
############################################################
# Convert the Neufeld & Kaufman 1993 (1993ApJ...418..263N)
# table into proper form
##############################

print( '\n==============================\n',
       'vibrational H2O\n==============================\n');
T  = array( [ 100,  200,  400,  1000,  2000,  4000 ] )
lgl0 = -log10( 1.03e-26 * T * exp( -47.5 / T**0.333 ) *
               exp( -2325 / T ) );
print( '  '.join( [ '%0.2f' % l for l in lgl0 ] ), '\n\n' );

a = [ [ 10.98, 11.05, 11.07, 10.88, 10.67, 10.67], 
[10.99, 11.05, 11.07, 10.88, 10.67, 10.67], 
[11.06, 11.09, 11.09, 10.88, 10.67, 10.67], 
[11.36, 11.31, 11.23, 10.93, 10.69, 10.68], 
[11.91, 11.82, 11.65, 11.15, 10.79, 10.72], 
[12.59, 12.50, 12.28, 11.61, 11.11, 10.90], 
[13.29, 13.20, 12.99, 12.25, 11.65, 11.30], 
[13.90, 13.90, 13.76, 13.01, 12.34, 11.88] ];
a = -array( a );
a = -log10( ( 10**a / exp( 2325 / T ) ) )
for a_s in a:
    print( '  '.join( [ '%0.2f' % l for l in a_s ] ) )


print( '\n==============================\n',
       'vibrational CO\n==============================\n');
T  = array( [ 100,  200,  400,  1000,  2000,  4000 ] );
lgl0 = -log10( 1.83e-26 * T * exp( -68 / T**0.333 ) *
             exp( -3080 / T ) );
print( '  '.join( [ '%0.2f' % l for l in lgl0 ] ), '\n\n' );

a = [[10.83, 10.82, 10.82, 10.80, 10.77, 10.79], 
[10.84, 10.83, 10.82, 10.80, 10.77, 10.79], 
[10.88, 10.86, 10.84, 10.81, 10.77, 10.80], 
[11.20, 11.11, 11.03, 10.92, 10.82, 10.82], 
[11.99, 11.86, 11.73, 11.42, 11.17, 11.04], 
[12.89, 12.77, 12.61, 12.18, 11.95, 11.75], 
[13.82, 13.70, 13.55, 13.03, 12.63, 12.31], 
[14.76, 14.65, 14.30, 13.93, 13.46, 13.09]]
a = -array( a );
T = array( [ 100, 200, 400, 1000, 2000, 4000 ] );
a = -log10( ( 10**a / exp( 3080 / T ) ) )
for a_s in a:
    print( '  '.join( [ '%0.2f' % l for l in a_s ] ) )

print( '\n==============================\n',
       'vibrational H2\n==============================\n');
T  = array( [ 100,  200,  400,  1000,  2000,  4000, 8e3,
              1e4 ] );
lgl0 = -log10( 1.19e-24 * sqrt( T )
               * exp( -18100 / ( T + 1190 ) )
               * exp( -5987 / T ) );
lglte = -log10( 1.1e-18 * exp( -6744 / T ) );
print( '  '.join( [ '%0.2f' % l for l in lgl0 ] ), '\n\n' )
print( '  '.join( [ '%0.2f' % l for l in lglte ] ) )
############################################################


 vibrational H2O

38.54  32.27  28.71  26.06  24.83  23.94 


21.08  16.10  13.59  11.89  11.17  10.92
21.09  16.10  13.59  11.89  11.17  10.92
21.16  16.14  13.61  11.89  11.17  10.92
21.46  16.36  13.75  11.94  11.19  10.93
22.01  16.87  14.17  12.16  11.29  10.97
22.69  17.55  14.80  12.62  11.61  11.15
23.39  18.25  15.51  13.26  12.15  11.55
24.00  18.95  16.28  14.02  12.84  12.13

 vibrational CO

43.49  35.18  30.50  27.04  25.46  24.34 


24.21  17.51  14.16  12.14  11.44  11.12
24.22  17.52  14.16  12.14  11.44  11.12
24.26  17.55  14.18  12.15  11.44  11.13
24.58  17.80  14.37  12.26  11.49  11.15
25.37  18.55  15.07  12.76  11.84  11.37
26.27  19.46  15.95  13.52  12.62  12.08
27.20  20.39  16.89  14.37  13.30  12.64
28.14  21.34  17.64  15.27  14.13  13.42

 vibrational H2

55.02  41.43  34.07  28.61  26.04  24.29  23.15  22.89 


47.25  32.60  25.28  20.89  19.42  18.69  18.32  18.25


In [6]:
############################################################
# Convert the cooling tables to readable format
##############################
file_input = 'mol_cooling.txt';

dat = configparser.RawConfigParser(  );
dat.optionxform = str;
# dat = {  };

sps  = None;
item = None
mode = None;

with open( file_input, 'r' ) as f:
    for line in f.readlines(  ):
        l = line.strip(  ).split(  );
        if  len( l ) < 1:
            continue;
        if  l[ 0 ] == '#':
            if len( l ) < 2:
                continue;
            if l[ 1 ] ==  'rotational' or \
               l[ 1 ] == 'vibrational' :
                mode  = l[ 1 ];
                sps   = l[ 2 ];
                label = 'reaction_' + mode + '_' + sps;
                dat [ label ] = { 'reactants' :  sps  ,
                                  'products'  :  sps  ,
                                  'mode'      : mode  ,
                                  'rate_type' : 'mol_cool' }
                continue;
            else:
                item = l[ 1 ];
            #
            continue;
        #
        if item[ 0 ] == 'm':
            res   = '-' + ' -'.join( l );
            item_ = item[ 1 : ];
        else:
            res   = ' '.join( l );
            item_ = item;
        if 'lg_' in item_:
            res = [ log( 10 )  * float(  s )
                    for s in res.split(  ) ];
            res = ' '.join( [ '%0.2f' %  s for s in res ] )
            item_ = item_.replace( 'lg_', 'ln_' );
        elif 'T' in item_:
            res = [ log( float( s ) )
                    for s in res.split(  ) ]
            res = ' '.join( [ '%0.2f' %  s for s in res ] )
            item_ = 'ln_T';
        #
        
        if  not item_ in dat[ label ]:
            dat[ label ][ item_ ] = '';
        #
        dat[ label ][ item_ ] += res
#

with open( 'mol_cooling.dat', 'w' ) as f:
    dat.write( f );
#
############################################################

In [3]:
############################################################
# Atomic cooling data
##############################
atomic_cool_dat = [  ];

def enroll_atom( sp, A, gu_gl, lam_micron,
                 T_nc, nc_n, nc_e ):
    hv = h * c / ( lam_micron * 1e-4 );
    atomic_cool_dat.append( [ sp, A, gu_gl, hv,
                              T_nc, nc_n, nc_e ] );
#

# https://physics.nist.gov/PhysRefData/ASD/lines_form.html
# Draine (2011) Pp. 192, Table 17.1

T_nc = [ 1e2, 5e3 ];
enroll_atom( 'C+',  2.29e-6, 4 / 2, 157.74,
             T_nc, [ 2.7e3, 1.5e3 ], [ 6.8, 40. ] );

enroll_atom( 'C',   7.93e-8, 3 / 1, 609.14,
             T_nc, [ 620.,   170. ], [ 76., 6.4 ] );
enroll_atom( 'C',   2.65e-7, 5 / 3, 370.42,
             T_nc, [ 720.,   150. ], [ 75., 6.3 ] );

enroll_atom( 'O',   8.91e-5, 3 / 5, 63.185,
             T_nc, [ 2.5e5, 4.9e4 ], [ 1.8e5, 4.8e4 ] );
enroll_atom( 'O',   1.75e-5, 1 / 3, 145.53,
             T_nc, [ 2.4e4, 8.6e3 ], [ 2.3e4, 5.8e3 ] );

enroll_atom( 'Si+', 2.10e-4, 4 / 2, 34.814,
             T_nc, [ 2.5e5, 1.2e5 ], [ 1.4e2, 1.5e3 ] );

enroll_atom( 'Si',  8.25e-6, 3 / 1, 129.68,
             T_nc, [ 4.8e4, 2.8e4 ], [ 2.9e4, 8.3e2 ] );
enroll_atom( 'Si',  4.21e-5, 5 / 3, 68.473,
             T_nc, [ 9.9e4, 3.6e4 ], [ 4.4e4, 1.9e3 ] )
############################################################

In [5]:
############################################################
# Convert the format of reaction database
##############################
cps = configparser.ConfigParser(  );

##############################
# 2-Body reactions

for i, l in enumerate( atomic_cool_dat ):
    sp, A, gu_gl, hv, T_nc, nc_n, nc_e = l;
    ldic = {  };
    ldic [ 'rate_type ' ] = 'atom_cool';
    ldic [ 'reactants ' ] = sp;
    ldic [ 'products  ' ] = ldic[ 'reactants ' ];

    lam = h * c / hv;
    # Draine (2011) eq. (6.20)
    f_osc = A / 0.6670 * lam**2 * gu_gl;
    tau0_ref = 0.7580 * ( 1 / 1e13 ) \
        * ( f_osc / 0.4164 ) * ( lam / 1215.7e-8 ) \
        * ( 10 / 1 );
    # tau0_ref for tau0 at N = 1e0 cm^-2 and b = 1 km/s
    pars = [ A, gu_gl, hv / kb, nc_n[ 0 ], nc_e[ 0 ],
             tau0_ref ];    
    ldic [ 'parameters' ] = ' '.join\
         ( [ '%0.2e' % p for p in pars ] );
    ldic [ 'energy    ' ] = '%0.2e' % hv;
    cps[ 'reaction_atomic_cool_%06d' % i ]  = ldic;
    #
#

with open( 'atom_cooling.dat', 'w' ) as f:
    cps.write( f );
#
############################################################